### Install Required Libraries

We need `transformers` for loading the model and tokenizer, and `accelerate` for efficient device mapping (handling GPU/CPU allocation).

In [3]:
!git clone https://huggingface.co/TheFireHacker/bubblspace-moe0.9b-a0.2b
%cd bubblspace-moe0.9b-a0.2b
!pip install -q -r requirements.txt

Cloning into 'bubblspace-moe0.9b-a0.2b'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 40 (delta 7), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (40/40), 649.24 KiB | 6.24 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/bubblspace-moe0.9b-a0.2b


### Load Model and Tokenizer

Next, we'll initialize the pipeline with the specified model `TheFireHacker/bubblspace-moe0.9b-a0.2b`. Since this is a Mixture-of-Experts (MoE) model, loading it with `device_map="auto"` is recommended to automatically utilize available GPU resources.

In [6]:
!apt-get install -y git-lfs -q
!git lfs pull

import torch
from safetensors.torch import load_file
from transformers import AutoTokenizer
from src.config import load_config
from src.model.moe import probe_grouped_mm
from src.model.transformer import Transformer

cfg = load_config("training_config.yaml")
cfg.model.moe.backend = "grouped_mm" if probe_grouped_mm() else "fallback"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("tokenizer")
model = Transformer(cfg.model).to(device)
model.load_state_dict(load_file("model.safetensors", device=device))
model.eval()

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 0s (31.7 MB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...


Transformer(
  (embed): Embedding(49280, 1024)
  (blocks): ModuleList(
    (0-15): 16 x TransformerBlock(
      (norm1): ZeroCenteredRMSNorm()
      (attn): Attention(
        (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (k_proj): Linear(in_features=1024, out_features=256, bias=False)
        (v_proj): Linear(in_features=1024, out_features=256, bias=False)
        (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (q_norm): ZeroCenteredRMSNorm()
        (k_norm): ZeroCenteredRMSNorm()
        (gate_proj): Linear(in_features=1024, out_features=1024, bias=False)
      )
      (norm2): ZeroCenteredRMSNorm()
      (moe): MoEFFN(
        (router): Linear(in_features=1024, out_features=64, bias=False)
      )
    )
  )
  (final_norm): ZeroCenteredRMSNorm()
)

### Generate Text

Now, let's write a simple helper function to generate responses from a prompt.

In [7]:
import torch.nn.functional as F

def generate_response(prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    real_vocab_size = len(tokenizer)
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    input_ids = torch.tensor([ids], device=device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            with torch.autocast(device_type=device, dtype=torch.bfloat16 if device == "cuda" else torch.float32):
                logits, _, _, _ = model(input_ids)
            next_logits = logits[0, -1, :real_vocab_size] / temperature
            top_vals, top_idx = next_logits.topk(top_k)
            probs = F.softmax(top_vals.float(), dim=-1)
            next_id = top_idx[torch.multinomial(probs, 1)]
            input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
            if next_id.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(input_ids[0].tolist())

prompt = "Albert Einstein what is his most notable work"
print(f"Prompt: {prompt}\n")
print(f"Response: {generate_response(prompt)}")

Prompt: Albert Einstein what is his most notable work

Response: Albert Einstein what is his most notable work, the famous Einstein's theory of relativity, was called.
What is the theory of relativity, and why will you be able to do what is right according to it. Einstein’s theory of relativity was first proposed in his 1905 paper "1000 years of history” by the english physicist john newbery, who was a british physicist who had been involved with the development of relativity in his.
An introduction to relativity, for example, states that the gravitational
